## What we will learn

1. S3 Buckets - Boto3
2. IAM Roles and Users
3. Complete Infrastructure of AWS Sagemaker - Training, EndpointsV

In [ ]:
import os
from pathlib import Path

import boto3
import pandas as pd
from dotenv import load_dotenv
from sagemaker.core.helper.session_helper import Session
from sklearn.model_selection import train_test_split

load_dotenv(Path.cwd() / ".env")

region = (
    os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION") or "us-east-1"
)
boto_sess = boto3.Session(region_name=region)
sm_boto3 = boto_sess.client("sagemaker")
sess = Session(boto_session=boto_sess)
bucket = os.environ.get("MOB_BUCKET_SAGEMAKER") or "mobbucketsagemaker"
print("Using bucket:", bucket)
print("Region:", region)


In [2]:
df = pd.read_csv("mob_price_classification_train.csv")
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [3]:
df.shape

(2000, 21)

In [4]:
df.isnull().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [5]:
df["price_range"].value_counts()

price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64

In [8]:
features = list(df.columns)
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi',
 'price_range']

In [11]:
label = features.pop(-1)
label

'price_range'

In [ ]:
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi']

In [ ]:
x = df[features]
y = df[label]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.15, random_state=0
)

In [15]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1700, 20)
(300, 20)
(1700,)
(300,)


In [16]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX = pd.DataFrame(X_test)
testX[label] = y_test

In [17]:
trainX

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
1452,1450,0,2.1,0,1,0,31,0.6,114,5,...,1573,1639,794,11,5,9,0,1,1,1
1044,1218,1,2.8,1,3,0,39,0.8,150,7,...,1122,1746,1667,10,0,12,0,0,0,1
1279,1602,0,0.6,0,12,0,58,0.4,170,1,...,1259,1746,3622,17,2,17,0,1,1,3
674,1034,0,2.6,1,2,1,45,0.3,190,3,...,182,1293,969,15,1,7,1,0,0,0
1200,530,0,2.4,0,1,0,32,0.3,88,6,...,48,1012,959,17,7,6,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,1224,1,1.6,0,9,0,33,1.0,157,1,...,522,563,3796,10,5,13,1,1,0,3
1216,1158,0,0.7,1,1,1,29,0.7,123,2,...,311,1796,1542,17,9,15,1,0,1,1
1653,1190,0,2.0,1,0,0,40,0.2,93,5,...,1399,1646,3610,13,7,9,0,0,1,3
559,1191,0,2.4,1,2,0,13,0.9,169,1,...,179,1813,1028,14,6,8,1,1,1,0


In [18]:
trainX.to_csv("train-V-1.csv", index=False)
testX.to_csv("test-V-1.csv", index=False)

In [19]:
bucket

'mobbucketsagemaker'

In [ ]:
## Send data to S3. Sagemaker will take the data for training from S3.
sk_prefix = "sagemaker/mobile_price_classification/sklearncontainer"
train_path = sess.upload_data(path="train-V-1.csv", bucket=bucket, key_prefix=sk_prefix)

test_path = sess.upload_data(path="test-V-1.csv", bucket=bucket, key_prefix=sk_prefix)

print(train_path)
print(test_path)

## Script used by AWS Sagemaker to train the model

In [27]:
%%writefile script.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
)
import sklearn
import joblib
import boto3
import pathlib
from io import StringIO
import argparse
import os
import numpy as np
import pandas as pd


def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))


if __name__ == "__main__":
    print("[Info] Extracting arguments")
    parser = argparse.ArgumentParser()

    # Hyperparameters
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--random_state", type=int, default=0)

    # Data, model, and output directories
    parser.add_argument("--model_dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    # Get arguments
    args, _ = parser.parse_known_args()

    print("Sklearn version: ", sklearn.__version__)
    print("Joblib version: ", joblib.__version__)

    print("[INFO] Reading data")
    print()
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing datasets")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print("Column order: ")
    print(features)
    print()

    print("Label column is: ", label)
    print()

    print("Data Shape: ")
    print()
    print("---- SHAPE OF TRAINING DATA (85%) ----")
    print(X_train.shape)
    print(y_train.shape)
    print()
    print("---- SHAPE OF TESTING DATA (15%) ----")
    print(X_test.shape)
    print(y_test.shape)
    print()

    print("Training RandomForest Model ....")
    print()
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        random_state=args.random_state,
        verbose=2,
        n_jobs=1,
    )

    model.fit(X_train, y_train)

    print()

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)

    print("Model saved at" + model_path)

    y_pred_test = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_rep = classification_report(y_test, y_pred_test)

    print()
    print("---- METRICS RESULTS FOR TESTING DATA ----")
    print()
    print("Total Rows are: ", X_test.shape[0])
    print("[TESTING] Model Accuracy is: ", test_acc)
    print("[TESTING] Testing Report: ")
    print(test_rep)


Overwriting script.py


In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train import ModelTrainer

FRAMEWORK_VERSION = "0.23-1"
ROLE = os.environ.get(
    "SAGEMAKER_ROLE",
    "arn:aws:iam::123456789012:role/sagemakeraccess",
)

training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version=FRAMEWORK_VERSION,
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="training",
)

source_code = SourceCode(
    source_dir=".",
    entry_script="script.py",
    ignore_patterns=[
        ".env",
        ".git",
        "__pycache__",
        ".DS_Store",
        ".cache",
        ".ipynb_checkpoints",
        ".ruff_cache",
        "venv",
        ".venv",
        "*.ipynb",
        "*.csv",
    ],
)

compute = Compute(
    instance_type="ml.m5.large",
    instance_count=1,
    enable_managed_spot_training=True,
)

stopping_condition = StoppingCondition(
    max_runtime_in_seconds=3600,
    max_wait_time_in_seconds=7200,
)

sklearn_trainer = ModelTrainer(
    training_image=training_image,
    sagemaker_session=sess,
    role=ROLE,
    source_code=source_code,
    compute=compute,
    stopping_condition=stopping_condition,
    hyperparameters={"n_estimators": 100, "random_state": 0},
    base_job_name="RF-custom-sklearn",
)

print("Training image:", training_image)
print("ModelTrainer ready (SageMaker SDK v3)")


## AWS Sagemaker entry point to execute the training script

In [ ]:
# Launch training job (SageMaker SDK v3)
train_data = InputData(channel_name="train", data_source=train_path)
test_data = InputData(channel_name="test", data_source=test_path)

sklearn_trainer.train(input_data_config=[train_data, test_data], wait=True)


## To get the model from S3

In [ ]:
training_job = sklearn_trainer._latest_training_job
training_job.wait(logs=False)

artifact = training_job.model_artifacts.s3_model_artifacts
# or, with boto3:
# artifact = sm_boto3.describe_training_job(
#     TrainingJobName=training_job.training_job_name
# )["ModelArtifacts"]["S3ModelArtifacts"]

In [ ]:
artifact